In [24]:
!pip install ibm-watsonx-ai --quiet

from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import ModelInference
import os
from getpass import getpass

if "WATSONX_APIKEY" not in os.environ:
    os.environ["WATSONX_APIKEY"] = getpass("Enter your watsonx API key: ")
# Watsonx.ai credentials (use your project's env values)
credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=os.environ.get("WATSONX_APIKEY")
)

project_id = os.environ.get("WATSONX_PROJECT_ID", "29455b9b-673a-4c96-83a4-6079538bc3c0")

In [25]:
model = ModelInference(
    model_id="ibm/granite-4-h-small",
    credentials=credentials,
    project_id=project_id,
    params={
        "max_tokens": 300,      # renamed from max_new_tokens
        "temperature": 0.3
    }
)
print("Model loaded:", model.model_id)

Model loaded: ibm/granite-4-h-small


In [26]:
academic_glossary = [
    {"term": "Federalism", "subject": "Political Science", "definition": "A system of government in which power is divided between a central authority and constituent political units."},
    {"term": "Shannon Entropy", "subject": "Computer Science", "definition": "A measure of uncertainty or randomness in information theory."},
    {"term": "Photosynthesis", "subject": "Biology", "definition": "The process by which green plants use sunlight to synthesize nutrients from carbon dioxide and water."}
]

def retrieve_context(query, subject_domain):
    # Simple keyword-based retrieval (real app uses FAISS + embeddings)
    matches = [g for g in academic_glossary if g["subject"].lower() == subject_domain.lower()]
    return matches

In [27]:
source_text = "Federalism ensures balanced power-sharing between state and central governments."
target_language = "Hindi"
subject_domain = "Political Science"

context = retrieve_context(source_text, subject_domain)
context_str = "\n".join([f"{c['term']}: {c['definition']}" for c in context])

prompt = f"""You are an academic translation assistant.
Use the following glossary context to ensure accurate, subject-specific translation.

Context:
{context_str}

Translate the following text into {target_language}, preserving technical accuracy:
"{source_text}"
"""

response = model.chat(messages=[{"role": "user", "content": prompt}])
translated_text = response["choices"][0]["message"]["content"]
print("Translated Output:\n", translated_text)

Translated Output:
 संघवाद राज्य और केंद्रीय सरकारों के बीच संतुलित शक्ति विभाजन सुनिश्चित करता है।


In [28]:
print("=== Translation Test Summary ===")
print(f"Source Text: {source_text}")
print(f"Target Language: {target_language}")
print(f"Subject Domain: {subject_domain}")
print(f"RAG Context Retrieved: {len(context)} entries")
print(f"Model Used: {model.model_id}")
print(f"Translated Output: {translated_text}")

=== Translation Test Summary ===
Source Text: Federalism ensures balanced power-sharing between state and central governments.
Target Language: Hindi
Subject Domain: Political Science
RAG Context Retrieved: 1 entries
Model Used: ibm/granite-4-h-small
Translated Output: संघवाद राज्य और केंद्रीय सरकारों के बीच संतुलित शक्ति विभाजन सुनिश्चित करता है।
